# Fine-tuned ResNet18 for EuroSAT

Stage 4 compares two preregistered fine-tuning strategies using the fixed seed-42 split. Model and epoch selection use validation macro-F1 only. The selected checkpoint is frozen before the single test evaluation.

**Split checksum:** `f97c4ec9a27435a932662d5a8b707255`

**Archival note:** Kaggle restarted between training and final evaluation. Candidate outputs below were consolidated from the verified history CSVs; checkpoint hashes, predictions, metrics, and the confusion matrix were independently validated after download.


## 1. Runtime and reproducibility controls


In [1]:
import hashlib
import platform
import random
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision

SEED = 42
EXPECTED_SPLIT_CHECKSUM = "f97c4ec9a27435a932662d5a8b707255"

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

print("Selected device:", DEVICE)

Python: 3.12.13
PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
Scikit-learn: 1.6.1
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
Selected device: cuda


In [2]:
%config HistoryManager.enabled = False

## 2. Dataset acquisition

Torchvision downloads the official EuroSAT RGB archive. Raw images are not committed to Git.


In [3]:
from torchvision.datasets import EuroSAT

DATA_BASE = Path("/kaggle/working/data")

eurosat = EuroSAT(
    root=DATA_BASE,
    download=True,
)

DATA_ROOT = DATA_BASE / "eurosat" / "2750"

print("Images:", len(eurosat))
print("Classes:", eurosat.classes)
print("Class mapping:", eurosat.class_to_idx)
print("Image directory:", DATA_ROOT)

Images: 27000
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Class mapping: {'AnnualCrop': 0, 'Forest': 1, 'HerbaceousVegetation': 2, 'Highway': 3, 'Industrial': 4, 'Pasture': 5, 'PermanentCrop': 6, 'Residential': 7, 'River': 8, 'SeaLake': 9}
Image directory: /kaggle/working/data/eurosat/2750


## 3. Fixed-split verification

The committed manifests are reused without modification. Test rows are checked for integrity only and are not used during selection.


In [4]:
SPLIT_URL = (
    "https://raw.githubusercontent.com/"
    "Cricdatahater/calibrated-eurosat/main/data/splits"
)

splits = {
    "train": pd.read_csv(f"{SPLIT_URL}/train.csv"),
    "validation": pd.read_csv(f"{SPLIT_URL}/validation.csv"),
    "test": pd.read_csv(f"{SPLIT_URL}/test.csv"),
}

expected_sizes = {
    "train": 18_900,
    "validation": 4_050,
    "test": 4_050,
}

for split_name, frame in splits.items():
    print(split_name, frame.shape)
    display(frame.head(2))

# Verify the original split-index file checksum.
split_index_url = f"{SPLIT_URL}/split_indices.json"
split_index_bytes = urllib.request.urlopen(split_index_url).read()
observed_checksum = hashlib.md5(split_index_bytes).hexdigest()

print("Expected checksum:", EXPECTED_SPLIT_CHECKSUM)
print("Observed checksum:", observed_checksum)

assert observed_checksum == EXPECTED_SPLIT_CHECKSUM

train (18900, 5)


,dataset_index,relative_path,class_name,class_index,split
0,20034,eurosat/2750/Residential/Residential_238.jpg,Residential,7,train
1,24872,eurosat/2750/SeaLake/SeaLake_1784.jpg,SeaLake,9,train


validation (4050, 5)


,dataset_index,relative_path,class_name,class_index,split
0,9029,eurosat/2750/Highway/Highway_1024.jpg,Highway,3,validation
1,13205,eurosat/2750/Industrial/Industrial_283.jpg,Industrial,4,validation


test (4050, 5)


,dataset_index,relative_path,class_name,class_index,split
0,16586,eurosat/2750/PermanentCrop/PermanentCrop_1526.jpg,PermanentCrop,6,test
1,24818,eurosat/2750/SeaLake/SeaLake_1735.jpg,SeaLake,9,test


Expected checksum: f97c4ec9a27435a932662d5a8b707255
Observed checksum: f97c4ec9a27435a932662d5a8b707255


In [5]:
all_indices = pd.concat(
    [frame["dataset_index"] for frame in splits.values()],
    ignore_index=True,
)

assert len(all_indices) == 27_000
assert all_indices.nunique() == 27_000

for split_name, expected_size in expected_sizes.items():
    assert len(splits[split_name]) == expected_size

expected_mapping = (
    pd.concat(splits.values(), ignore_index=True)
    [["class_name", "class_index"]]
    .drop_duplicates()
    .set_index("class_name")["class_index"]
    .to_dict()
)

assert eurosat.class_to_idx == expected_mapping

def resolve_path(row):
    return (
        DATA_ROOT
        / row["class_name"]
        / Path(row["relative_path"]).name
    )

for split_name, frame in splits.items():
    missing_files = [
        resolve_path(row)
        for _, row in frame.iterrows()
        if not resolve_path(row).is_file()
    ]

    print(f"{split_name}: {len(missing_files)} missing files")
    assert not missing_files

print("Dataset and fixed-split validation passed.")

train: 0 missing files
validation: 0 missing files
test: 0 missing files
Dataset and fixed-split validation passed.


## 4. Transforms, datasets, and data loaders

Training uses random resized crops and horizontal/vertical flips. Validation and test preprocessing is deterministic and matches `IMAGENET1K_V1`.


In [6]:
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights
from torchvision.transforms import InterpolationMode

BATCH_SIZE = 64
NUM_WORKERS = 2

weights = ResNet18_Weights.IMAGENET1K_V1

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.85, 1.0),
        interpolation=InterpolationMode.BILINEAR,
        antialias=True,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

evaluation_transform = weights.transforms()


class ManifestDataset(Dataset):
    def __init__(self, manifest, data_root, transform):
        self.manifest = manifest.reset_index(drop=True)
        self.data_root = Path(data_root)
        self.transform = transform

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, index):
        row = self.manifest.iloc[index]

        image_path = (
            self.data_root
            / row["class_name"]
            / Path(row["relative_path"]).name
        )

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        return (
            image,
            int(row["class_index"]),
            int(row["dataset_index"]),
        )


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_dataset = ManifestDataset(
    splits["train"], DATA_ROOT, train_transform
)

validation_dataset = ManifestDataset(
    splits["validation"], DATA_ROOT, evaluation_transform
)

test_dataset = ManifestDataset(
    splits["test"], DATA_ROOT, evaluation_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    generator=loader_generator,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
)

images, labels, dataset_indices = next(iter(train_loader))

print("Training samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))
print("Test samples:", len(test_dataset))
print("Image batch:", images.shape)
print("Label batch:", labels.shape)
print("Index batch:", dataset_indices.shape)

assert images.shape == (BATCH_SIZE, 3, 224, 224)
assert labels.min() >= 0
assert labels.max() < 10

print("Transforms and data loaders passed the smoke test.")

Training samples: 18900
Validation samples: 4050
Test samples: 4050
Image batch: torch.Size([64, 3, 224, 224])
Label batch: torch.Size([64])
Index batch: torch.Size([64])
Transforms and data loaders passed the smoke test.


## 5. Model construction and head-only warm-up

An ImageNet-pretrained ResNet18 receives a new 10-class head. The backbone remains frozen for three warm-up epochs; only `fc.weight` and `fc.bias` are trainable. Both candidates start from the identical epoch-3 checkpoint.


In [7]:
import torch.nn as nn
from torchvision.models import resnet18

NUM_CLASSES = len(eurosat.classes)

# Reset randomness because the loader smoke test consumed random values.
seed_everything(SEED)
loader_generator.manual_seed(SEED)

model = resnet18(weights=weights)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)

# Freeze the complete network.
for parameter in model.parameters():
    parameter.requires_grad = False

# Unfreeze only the new classifier.
for parameter in model.fc.parameters():
    parameter.requires_grad = True

model = model.to(DEVICE)

trainable_names = [
    name for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel() for parameter in model.parameters()
)

print("Trainable parameter names:", trainable_names)
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Total parameters: {total_parameters:,}")

assert trainable_names == ["fc.weight", "fc.bias"]
assert model.fc.out_features == 10

print("Head-only freeze-state verification passed.")

Trainable parameter names: ['fc.weight', 'fc.bias']
Trainable parameters: 5,130
Total parameters: 11,181,642
Head-only freeze-state verification passed.


### Training and metric helpers


In [8]:
from contextlib import nullcontext

from sklearn.metrics import accuracy_score, f1_score, log_loss
from tqdm.auto import tqdm

USE_AMP = DEVICE.type == "cuda"


def freeze_inactive_batch_norm(model):
    """
    Prevent BatchNorm statistics from changing in frozen sections.
    """
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            parameters = list(module.parameters())

            if parameters and not any(p.requires_grad for p in parameters):
                module.eval()


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
):
    model.train()
    freeze_inactive_batch_norm(model)

    running_loss = 0.0
    correct = 0
    sample_count = 0

    progress = tqdm(loader, desc="Training", leave=False)

    for images, labels, _ in progress:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )
        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()
        sample_count += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return {
        "train_loss": running_loss / sample_count,
        "train_accuracy": correct / sample_count,
    }


@torch.inference_mode()
def evaluate_model(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    sample_count = 0

    all_labels = []
    all_probabilities = []
    all_indices = []

    for images, labels, dataset_indices in tqdm(
        loader,
        desc="Validation",
        leave=False,
    ):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )
        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        sample_count += batch_size

        all_labels.append(labels.cpu())
        all_probabilities.append(probabilities.cpu())
        all_indices.append(dataset_indices.cpu())

    y_true = torch.cat(all_labels).numpy()
    probabilities = torch.cat(all_probabilities).numpy()
    dataset_indices = torch.cat(all_indices).numpy()
    predictions = probabilities.argmax(axis=1)

    metrics = {
        "validation_loss": running_loss / sample_count,
        "validation_accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "validation_macro_f1": f1_score(
            y_true,
            predictions,
            average="macro",
        ),
        "validation_log_loss": log_loss(
            y_true,
            probabilities,
            labels=list(range(NUM_CLASSES)),
        ),
    }

    return metrics, y_true, probabilities, dataset_indices

In [9]:
OUTPUT_DIR = Path("/kaggle/working/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WARMUP_CHECKPOINT = OUTPUT_DIR / "head_warmup_best.pth"

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

warmup_history = []
best_macro_f1 = -float("inf")

for epoch in range(1, 4):
    training_metrics = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler,
    )

    validation_metrics, _, _, _ = evaluate_model(
        model,
        validation_loader,
        criterion,
    )

    epoch_metrics = {
        "epoch": epoch,
        **training_metrics,
        **validation_metrics,
    }

    warmup_history.append(epoch_metrics)

    print(
        f"Epoch {epoch}/3 | "
        f"train loss: {training_metrics['train_loss']:.4f} | "
        f"train accuracy: {training_metrics['train_accuracy']:.4f} | "
        f"validation loss: "
        f"{validation_metrics['validation_loss']:.4f} | "
        f"validation accuracy: "
        f"{validation_metrics['validation_accuracy']:.4f} | "
        f"validation macro-F1: "
        f"{validation_metrics['validation_macro_f1']:.4f}"
    )

    if (
        validation_metrics["validation_macro_f1"]
        > best_macro_f1
    ):
        best_macro_f1 = validation_metrics[
            "validation_macro_f1"
        ]

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "validation_metrics": validation_metrics,
                "weights": "IMAGENET1K_V1",
                "split_checksum": EXPECTED_SPLIT_CHECKSUM,
                "seed": SEED,
                "num_classes": NUM_CLASSES,
            },
            WARMUP_CHECKPOINT,
        )

        print("Saved new best warm-up checkpoint.")

warmup_history = pd.DataFrame(warmup_history)
display(warmup_history)

print("Best validation macro-F1:", best_macro_f1)
print("Checkpoint:", WARMUP_CHECKPOINT)

Training:   0%|          | 0/296 [00:00<?, ?it/s]

Validation:   0%|          | 0/64 [00:00<?, ?it/s]

Epoch 1/3 | train loss: 0.5510 | train accuracy: 0.8570 | validation loss: 0.2898 | validation accuracy: 0.9131 | validation macro-F1: 0.9101
Saved new best warm-up checkpoint.


Training:   0%|          | 0/296 [00:00<?, ?it/s]

Validation:   0%|          | 0/64 [00:00<?, ?it/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fdf849b2ac0>
Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7fdf849b2ac0>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
self._shutdown_workers()Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    self._shutdown_workers()    
if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

     if w.is_alive():  
    Exception ignored in:   Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7fdf849b2ac0> <function _MultiProcessingDataLoaderIter.__del__ at 0x7fdf849b2ac0>^
 
^ Traceback (most recent 

Validation:   0%|          | 0/64 [00:00<?, ?it/s]

Epoch 3/3 | train loss: 0.2084 | train accuracy: 0.9337 | validation loss: 0.2194 | validation accuracy: 0.9281 | validation macro-F1: 0.9260
Saved new best warm-up checkpoint.


,epoch,train_loss,train_accuracy,validation_loss,validation_accuracy,validation_macro_f1,validation_log_loss
0,1,0.551022,0.856984,0.289786,0.913086,0.910109,0.289788
1,2,0.243236,0.926402,0.237837,0.925679,0.922664,0.237838
2,3,0.208415,0.933651,0.219392,0.928148,0.925971,0.219391


Best validation macro-F1: 0.9259705985525951
Checkpoint: /kaggle/working/models/head_warmup_best.pth


## 6. Validation-only fine-tuning candidates

- Candidate A trains `layer4` and `fc`.
- Candidate B trains the full network with discriminative learning rates.

Both use the registered optimizer, scheduler, maximum epoch count, and early-stopping rule. The test loader is not evaluated here.


In [10]:
import gc

RESULTS_DIR = Path(
    "/kaggle/working/results/fine_tuned_resnet18"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert WARMUP_CHECKPOINT.is_file(), (
    "Warm-up checkpoint is missing. Re-run the warm-up cell."
)


def build_candidate(candidate_name):
    candidate_model = resnet18(weights=None)
    candidate_model.fc = nn.Linear(
        candidate_model.fc.in_features,
        NUM_CLASSES,
    )

    warmup_checkpoint = torch.load(
        WARMUP_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    candidate_model.load_state_dict(
        warmup_checkpoint["model_state_dict"]
    )

    if candidate_name == "candidate_a":
        # Freeze everything, then unfreeze layer4 and fc.
        for parameter in candidate_model.parameters():
            parameter.requires_grad = False

        for parameter in candidate_model.layer4.parameters():
            parameter.requires_grad = True

        for parameter in candidate_model.fc.parameters():
            parameter.requires_grad = True

        optimizer = torch.optim.AdamW(
            [
                {
                    "params": candidate_model.layer4.parameters(),
                    "lr": 1e-4,
                },
                {
                    "params": candidate_model.fc.parameters(),
                    "lr": 5e-4,
                },
            ],
            weight_decay=1e-4,
        )

    elif candidate_name == "candidate_b":
        # Full-network fine-tuning.
        for parameter in candidate_model.parameters():
            parameter.requires_grad = True

        early_parameters = (
            list(candidate_model.conv1.parameters())
            + list(candidate_model.bn1.parameters())
            + list(candidate_model.layer1.parameters())
            + list(candidate_model.layer2.parameters())
            + list(candidate_model.layer3.parameters())
        )

        optimizer = torch.optim.AdamW(
            [
                {
                    "params": early_parameters,
                    "lr": 1e-5,
                },
                {
                    "params": candidate_model.layer4.parameters(),
                    "lr": 5e-5,
                },
                {
                    "params": candidate_model.fc.parameters(),
                    "lr": 5e-4,
                },
            ],
            weight_decay=1e-4,
        )

    else:
        raise ValueError(f"Unknown candidate: {candidate_name}")

    return candidate_model.to(DEVICE), optimizer

In [11]:
MAX_EPOCHS = 12
EARLY_STOPPING_PATIENCE = 4


def run_candidate(candidate_name):
    # Ensures both candidates receive the same initial random sequence.
    seed_everything(SEED)
    loader_generator.manual_seed(SEED)

    candidate_model, candidate_optimizer = build_candidate(
        candidate_name
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        candidate_optimizer,
        mode="min",
        factor=0.2,
        patience=2,
    )

    candidate_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )

    checkpoint_path = (
        RESULTS_DIR / f"{candidate_name}_best.pth"
    )
    history_path = (
        RESULTS_DIR / f"{candidate_name}_history.csv"
    )

    history = []
    best_macro_f1 = -float("inf")
    best_log_loss = float("inf")
    best_epoch = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        training_metrics = train_one_epoch(
            candidate_model,
            train_loader,
            criterion,
            candidate_optimizer,
            candidate_scaler,
        )

        validation_metrics, _, _, _ = evaluate_model(
            candidate_model,
            validation_loader,
            criterion,
        )

        scheduler.step(
            validation_metrics["validation_log_loss"]
        )

        learning_rates = [
            group["lr"]
            for group in candidate_optimizer.param_groups
        ]

        record = {
            "candidate": candidate_name,
            "epoch": epoch,
            **training_metrics,
            **validation_metrics,
            "learning_rates": str(learning_rates),
        }

        history.append(record)
        pd.DataFrame(history).to_csv(
            history_path,
            index=False,
        )

        current_f1 = validation_metrics[
            "validation_macro_f1"
        ]
        current_log_loss = validation_metrics[
            "validation_log_loss"
        ]

        improved = (
            current_f1 > best_macro_f1 + 1e-12
            or (
                abs(current_f1 - best_macro_f1) <= 1e-12
                and current_log_loss < best_log_loss
            )
        )

        if improved:
            best_macro_f1 = current_f1
            best_log_loss = current_log_loss
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(
                {
                    "candidate": candidate_name,
                    "epoch": epoch,
                    "model_state_dict":
                        candidate_model.state_dict(),
                    "optimizer_state_dict":
                        candidate_optimizer.state_dict(),
                    "validation_metrics":
                        validation_metrics,
                    "split_checksum":
                        EXPECTED_SPLIT_CHECKSUM,
                    "seed": SEED,
                },
                checkpoint_path,
            )

            status = "saved"
        else:
            epochs_without_improvement += 1
            status = (
                f"no improvement "
                f"({epochs_without_improvement}/"
                f"{EARLY_STOPPING_PATIENCE})"
            )

        print(
            f"{candidate_name} | epoch {epoch} | "
            f"train loss {training_metrics['train_loss']:.4f} | "
            f"val loss "
            f"{validation_metrics['validation_loss']:.4f} | "
            f"val accuracy "
            f"{validation_metrics['validation_accuracy']:.4f} | "
            f"val macro-F1 {current_f1:.4f} | "
            f"{status}"
        )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print("Early stopping activated.")
            break

    summary = {
        "candidate": candidate_name,
        "best_epoch": best_epoch,
        "best_validation_macro_f1": best_macro_f1,
        "best_validation_log_loss": best_log_loss,
        "checkpoint": str(checkpoint_path),
        "history": str(history_path),
    }

    del candidate_model
    del candidate_optimizer
    torch.cuda.empty_cache()
    gc.collect()

    return summary

### Candidate A: partial fine-tuning


In [12]:
candidate_a_summary = run_candidate("candidate_a")
candidate_a_summary

candidate_a | epoch 1 | train loss 0.2002 | val loss 0.1457 | val accuracy 0.9551 | val macro-F1 0.9533 | saved
candidate_a | epoch 2 | train loss 0.1095 | val loss 0.1296 | val accuracy 0.9600 | val macro-F1 0.9584 | saved
candidate_a | epoch 3 | train loss 0.0850 | val loss 0.1152 | val accuracy 0.9649 | val macro-F1 0.9637 | saved
candidate_a | epoch 4 | train loss 0.0682 | val loss 0.0996 | val accuracy 0.9696 | val macro-F1 0.9685 | saved
candidate_a | epoch 5 | train loss 0.0497 | val loss 0.1535 | val accuracy 0.9615 | val macro-F1 0.9598 | no improvement (1/4)
candidate_a | epoch 6 | train loss 0.0479 | val loss 0.0940 | val accuracy 0.9726 | val macro-F1 0.9717 | saved
candidate_a | epoch 7 | train loss 0.0459 | val loss 0.0989 | val accuracy 0.9746 | val macro-F1 0.9738 | saved
candidate_a | epoch 8 | train loss 0.0342 | val loss 0.1166 | val accuracy 0.9733 | val macro-F1 0.9724 | no improvement (1/4)
candidate_a | epoch 9 | train loss 0.0299 | val loss 0.1257 | val accuracy

{'candidate': 'candidate_a', 'best_epoch': 12, 'best_validation_macro_f1': 0.977565113695982, 'best_validation_log_loss': 0.0885230745751517, 'checkpoint': '/kaggle/working/results/fine_tuned_resnet18/candidate_a_best.pth', 'history': '/kaggle/working/results/fine_tuned_resnet18/candidate_a_history.csv'}

### Candidate B: full fine-tuning


In [13]:
candidate_b_summary = run_candidate("candidate_b")
candidate_b_summary

candidate_b | epoch 1 | train loss 0.2587 | val loss 0.1381 | val accuracy 0.9548 | val macro-F1 0.9530 | saved
candidate_b | epoch 2 | train loss 0.1065 | val loss 0.1197 | val accuracy 0.9640 | val macro-F1 0.9628 | saved
candidate_b | epoch 3 | train loss 0.0791 | val loss 0.0947 | val accuracy 0.9696 | val macro-F1 0.9685 | saved
candidate_b | epoch 4 | train loss 0.0686 | val loss 0.1016 | val accuracy 0.9681 | val macro-F1 0.9672 | no improvement (1/4)
candidate_b | epoch 5 | train loss 0.0521 | val loss 0.1261 | val accuracy 0.9652 | val macro-F1 0.9642 | no improvement (2/4)
candidate_b | epoch 6 | train loss 0.0499 | val loss 0.0925 | val accuracy 0.9738 | val macro-F1 0.9732 | saved
candidate_b | epoch 7 | train loss 0.0395 | val loss 0.0744 | val accuracy 0.9775 | val macro-F1 0.9770 | saved
candidate_b | epoch 8 | train loss 0.0390 | val loss 0.0928 | val accuracy 0.9775 | val macro-F1 0.9769 | no improvement (1/4)
candidate_b | epoch 9 | train loss 0.0322 | val loss 0.0983

{'candidate': 'candidate_b', 'best_epoch': 7, 'best_validation_macro_f1': 0.97700273761596, 'best_validation_log_loss': 0.07439034946798424, 'checkpoint': '/kaggle/working/results/fine_tuned_resnet18/candidate_b_best.pth', 'history': '/kaggle/working/results/fine_tuned_resnet18/candidate_b_history.csv'}

### Validation comparison


In [14]:
candidate_comparison = pd.DataFrame([
    candidate_a_summary,
    candidate_b_summary,
])

display(candidate_comparison)

## 7. Frozen model selection

Candidate A is selected because it has the higher validation macro-F1. Candidate B has lower log loss, but log loss is only a tie-breaker and macro-F1 is not tied. Candidate A at epoch 12 and its SHA-256 hash are frozen before test access.


In [15]:
import json

SELECTED_CANDIDATE = "candidate_a"
SELECTED_EPOCH = 12

SELECTED_CHECKPOINT = (
    RESULTS_DIR / "candidate_a_best.pth"
)

assert SELECTED_CHECKPOINT.is_file()

selected_checkpoint = torch.load(
    SELECTED_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

assert selected_checkpoint["candidate"] == SELECTED_CANDIDATE
assert selected_checkpoint["epoch"] == SELECTED_EPOCH
assert (
    selected_checkpoint["split_checksum"]
    == EXPECTED_SPLIT_CHECKSUM
)

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

checkpoint_sha256 = sha256_file(SELECTED_CHECKPOINT)

frozen_configuration = {
    "selected_candidate": SELECTED_CANDIDATE,
    "selected_epoch": SELECTED_EPOCH,
    "selection_metric": "validation_macro_f1",
    "validation_macro_f1": selected_checkpoint[
        "validation_metrics"
    ]["validation_macro_f1"],
    "validation_log_loss": selected_checkpoint[
        "validation_metrics"
    ]["validation_log_loss"],
    "checkpoint_sha256": checkpoint_sha256,
    "split_checksum": EXPECTED_SPLIT_CHECKSUM,
    "seed": SEED,
}

with (
    RESULTS_DIR / "frozen_selection.json"
).open("w") as file:
    json.dump(frozen_configuration, file, indent=2)

print(json.dumps(frozen_configuration, indent=2))

{
  "selected_candidate": "candidate_a",
  "selected_epoch": 12,
  "selection_metric": "validation_macro_f1",
  "validation_macro_f1": 0.977565113695982,
  "validation_log_loss": 0.0885230745751517,
  "checkpoint_sha256": "3f6a701aca082fa675ba229ddfbae0139346e3cbff84ea02ff0617ab73c5d657",
  "split_checksum": "f97c4ec9a27435a932662d5a8b707255",
  "seed": 42
}


## 8. Single locked test evaluation

The selected checkpoint is evaluated once on the untouched test split. No model or hyperparameter decision is changed in response to the result.


In [16]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)

# A single-process loader avoids Kaggle worker-cleanup warnings.
final_test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

final_model = resnet18(weights=None)
final_model.fc = nn.Linear(
    final_model.fc.in_features,
    NUM_CLASSES,
)

final_model.load_state_dict(
    selected_checkpoint["model_state_dict"]
)

final_model = final_model.to(DEVICE)
final_model.eval()

all_test_labels = []
all_test_probabilities = []
all_test_indices = []

test_loss_sum = 0.0
test_sample_count = 0

with torch.inference_mode():
    for images, labels, indices in tqdm(
        final_test_loader,
        desc="Final test evaluation",
    ):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):
            logits = final_model(images)
            loss = criterion(logits, labels)

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        batch_size = labels.size(0)
        test_loss_sum += loss.item() * batch_size
        test_sample_count += batch_size

        all_test_labels.append(labels.cpu())
        all_test_probabilities.append(
            probabilities.cpu()
        )
        all_test_indices.append(indices.cpu())

y_test = torch.cat(all_test_labels).numpy()
test_probabilities = torch.cat(
    all_test_probabilities
).numpy()
test_indices = torch.cat(all_test_indices).numpy()
test_predictions = test_probabilities.argmax(axis=1)

assert len(y_test) == 4_050
assert len(np.unique(test_indices)) == 4_050
assert test_probabilities.shape == (4_050, 10)

Final test evaluation:   0%|          | 0/64 [00:00<?, ?it/s]

### Test metrics and calibration


In [17]:
def multiclass_brier_score(y_true, probabilities):
    one_hot = np.eye(
        probabilities.shape[1]
    )[y_true]

    return np.mean(
        np.sum(
            (probabilities - one_hot) ** 2,
            axis=1,
        )
    )


def calculate_ece(
    y_true,
    probabilities,
    n_bins=15,
):
    predictions = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    correct = predictions == y_true

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    records = []
    ece = 0.0

    for bin_index in range(n_bins):
        lower = edges[bin_index]
        upper = edges[bin_index + 1]

        if bin_index == 0:
            mask = (
                (confidence >= lower)
                & (confidence <= upper)
            )
        else:
            mask = (
                (confidence > lower)
                & (confidence <= upper)
            )

        count = int(mask.sum())

        if count:
            bin_accuracy = float(correct[mask].mean())
            bin_confidence = float(
                confidence[mask].mean()
            )

            ece += (
                count / len(y_true)
            ) * abs(bin_accuracy - bin_confidence)
        else:
            bin_accuracy = np.nan
            bin_confidence = np.nan

        records.append({
            "bin": bin_index + 1,
            "lower": lower,
            "upper": upper,
            "count": count,
            "accuracy": bin_accuracy,
            "mean_confidence": bin_confidence,
        })

    return float(ece), pd.DataFrame(records)


test_ece, calibration_bins = calculate_ece(
    y_test,
    test_probabilities,
)

test_metrics = {
    "test_cross_entropy":
        test_loss_sum / test_sample_count,
    "accuracy": accuracy_score(
        y_test,
        test_predictions,
    ),
    "macro_f1": f1_score(
        y_test,
        test_predictions,
        average="macro",
    ),
    "log_loss": log_loss(
        y_test,
        test_probabilities,
        labels=list(range(NUM_CLASSES)),
    ),
    "multiclass_brier_score":
        multiclass_brier_score(
            y_test,
            test_probabilities,
        ),
    "expected_calibration_error": test_ece,
    "ece_bins": 15,
}

print(json.dumps(test_metrics, indent=2))

{
  "test_cross_entropy": 0.07406974015378014,
  "accuracy": 0.9795061728395061,
  "macro_f1": 0.9785400949175062,
  "log_loss": 0.07407171916617017,
  "multiclass_brier_score": 0.03343749292798481,
  "expected_calibration_error": 0.0122128208919808,
  "ece_bins": 15
}


### Artifact export


In [18]:
class_names = eurosat.classes

per_class_report = pd.DataFrame(
    classification_report(
        y_test,
        test_predictions,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
).transpose()

confusion = pd.DataFrame(
    confusion_matrix(
        y_test,
        test_predictions,
    ),
    index=class_names,
    columns=class_names,
)

prediction_table = pd.DataFrame({
    "dataset_index": test_indices,
    "true_class_index": y_test,
    "true_class_name": [
        class_names[index] for index in y_test
    ],
    "predicted_class_index": test_predictions,
    "predicted_class_name": [
        class_names[index]
        for index in test_predictions
    ],
    "confidence": test_probabilities.max(axis=1),
})

for class_index, class_name in enumerate(class_names):
    prediction_table[
        f"probability_{class_name}"
    ] = test_probabilities[:, class_index]

with (RESULTS_DIR / "test_metrics.json").open("w") as file:
    json.dump(test_metrics, file, indent=2)

per_class_report.to_csv(
    RESULTS_DIR / "test_per_class_metrics.csv"
)

confusion.to_csv(
    RESULTS_DIR / "test_confusion_matrix.csv"
)

calibration_bins.to_csv(
    RESULTS_DIR / "test_calibration_bins.csv",
    index=False,
)

prediction_table.to_csv(
    RESULTS_DIR / "test_predictions.csv",
    index=False,
)

print("Saved results to:", RESULTS_DIR)

Saved results to: /kaggle/working/results/fine_tuned_resnet18


## 9. Findings and limitations

- Test accuracy: **0.979506**; macro-F1: **0.978540**.
- Test log loss: **0.074072**; multiclass Brier score: **0.033437**; 15-bin ECE: **0.012213**.
- Fine-tuning improves all five reported test metrics over the frozen ResNet18 baseline.
- Candidate A changes fewer pretrained parameters than full fine-tuning while attaining slightly higher validation macro-F1.
- The fixed image-level split does not establish generalization to unseen geographic regions.
- This is a controlled project-specific recipe until the target research paper and its published settings are identified and audited.
